In [6]:
import threading
import joblib
import pandas as pd
from flask import Flask, request, render_template_string

# Load the trained model (must be in the same folder as this notebook)
model = joblib.load("aqi_regressor.joblib")
FEATURES = ["PM2_5_ugm3", "PM10_ugm3", "NO2_ugm3", "SO2_ugm3", "O3_ugm3", "CO_ugm3"]

app = Flask(__name__)

PAGE = """
<!DOCTYPE html>
<html>
<head>
<meta charset="UTF-8">
<title>Air Quality Index Predictor</title>
<style>
  body { font-family: Arial, sans-serif; background: #f3f4f6; margin: 0; padding: 40px; }
  .card { max-width: 480px; margin: auto; background: #fff; border-radius: 12px;
          padding: 30px; box-shadow: 0 2px 8px rgba(0,0,0,0.1); }
  h1 { font-size: 22px; margin-bottom: 4px; }
  p.sub { color: #6b7280; margin-top: 0; font-size: 14px; }
  label { display: block; margin-top: 12px; font-size: 14px; font-weight: bold; color: #374151; }
  input { width: 100%; padding: 8px; margin-top: 4px; border: 1px solid #d1d5db;
          border-radius: 6px; box-sizing: border-box; }
  button { margin-top: 20px; width: 100%; padding: 10px; background: #6366f1;
           color: white; border: none; border-radius: 6px; font-size: 15px; cursor: pointer; }
  .result { margin-top: 20px; padding: 15px; border-radius: 8px; color: #fff; text-align: center; }
  .aqi { font-size: 32px; font-weight: bold; }
  .error { background: #fef2f2; border: 1px solid #ef4444; color: #b91c1c;
           padding: 15px; border-radius: 8px; margin-top: 20px; }
</style>
</head>
<body>
  <div class="card">
    <h1>Air Quality Index Predictor</h1>
    <p class="sub">Enter pollutant readings to get the predicted US AQI (AdaBoost model)</p>
    <form action="/predict" method="POST">
      <label>PM2.5 (µg/m³)</label><input type="number" step="any" name="PM2.5" required>
      <label>PM10 (µg/m³)</label><input type="number" step="any" name="PM10" required>
      <label>NO2 (µg/m³)</label><input type="number" step="any" name="NO2" required>
      <label>SO2 (µg/m³)</label><input type="number" step="any" name="SO2" required>
      <label>O3 (µg/m³)</label><input type="number" step="any" name="O3" required>
      <label>CO (µg/m³)</label><input type="number" step="any" name="CO" required>
      <button type="submit">Predict AQI</button>
    </form>
    {% if error %}
      <div class="error">{{ error }}</div>
    {% elif prediction is not none %}
      <div class="result" style="background: {{ color }};">
        <div class="aqi">AQI: {{ prediction }}</div>
        <div>{{ category }}</div>
      </div>
    {% endif %}
  </div>
</body>
</html>
"""


def aqi_category(v):
    if v <= 50:
        return "Good", "#2e9e4f"
    elif v <= 100:
        return "Moderate", "#d4a017"
    elif v <= 150:
        return "Unhealthy for Sensitive Groups", "#ff7e00"
    elif v <= 200:
        return "Unhealthy", "#e03131"
    elif v <= 300:
        return "Very Unhealthy", "#8f3f97"
    return "Hazardous", "#7e0023"


@app.route("/", methods=["GET"])
def home():
    return render_template_string(PAGE, prediction=None, error=None)


@app.route("/predict", methods=["POST"])
def predict():
    try:
        values = [float(request.form[k]) for k in ["PM2.5", "PM10", "NO2", "SO2", "O3", "CO"]]
        X = pd.DataFrame([values], columns=FEATURES)
        aqi = round(float(model.predict(X)[0]), 1)
        category, color = aqi_category(aqi)
        return render_template_string(PAGE, prediction=aqi, category=category,
                                      color=color, error=None)
    except Exception as e:
        return render_template_string(PAGE, prediction=None, error=f"Error: {e}")


def run_app():
    app.run(port=5000, debug=False, use_reloader=False)


threading.Thread(target=run_app, daemon=True).start()
print("Server started. Open http://127.0.0.1:5000 in your browser")

Server started. Open http://127.0.0.1:5000 in your browser
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [24/Oct/2026 18:09:33] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [24/Oct/2026 18:09:33] "GET /favicon.ico HTTP/1.1" 404 -
